# Adicionando funções externas utilizando LangChain

LangChain é um framework para criação de aplicações de IA e naturalmente ele possui ferramentas para facilitar a criação de funções externas para serem passadas a API da OpenAI. Para a utilização do LangChain, será necessário entendermos brevemente uma outra biblioteca de Python chamada Pydantic, uma biblioteca para validação de dados que facilita a construção de estruturas de dados mais robustas.

## Introduzindo pydantic

### Como criamos uma estrutura nova sem pydantic

Sem utilizar pydantic, podemos criar uma estrutura de dados utilizando funções da seguinte forma:

In [1]:
class Pessoa:
	def __init__(self, nome: str, idade: int, peso: float) -> None:
		self.nome = nome
		self.idade = idade
		self.peso = peso

Neste caso, criamos uma função que representa uma pessoa e tem os seguintes atributos: nome, idade e peso.

In [3]:
bruno = Pessoa("Bruno", 32, 90)
bruno

In [4]:
bruno.idade

32

In [5]:
bruno = Pessoa("Bruno", 32, 'ashdbadgvuya')
bruno

In [6]:
bruno.peso

'ashdbadgvuya'

## Como criamos uma estrutura nova usando Pydantic

A sintaxe de pydantic acaba sendo bem mais simples para a criação de classes de dados, ao compararmos com a criação de classes comuns de Python. Nela, temos que cuidar com a definição do tipo de cada atributo, pois eles serão utilizados para validar se os dados fornecidos estão corretos.

In [7]:
from pydantic import BaseModel

class pydPessoa(BaseModel):
	nome: str
	idade: int
	peso: float

In [8]:
bruno = pydPessoa(nome="Bruno", idade=32, peso=90.5)
bruno

pydPessoa(nome='Bruno', idade=32, peso=90.5)

In [9]:
bruno.nome

'Bruno'

O interessante é vermos que pydantic fornece uma validação automática de dados. Isso garante uma integridade muito maior em aplicações mais complexas.

In [10]:
bruno = pydPessoa(nome="Bruno", idade=32, peso='ashdbadgvuya')

ValidationError: 1 validation error for pydPessoa
peso
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='ashdbadgvuya', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/float_parsing

In [11]:
bruno = pydPessoa(nome="Bruno", idade=32, peso="90.5")
bruno.peso

90.5

Podemos fazer um nesting de classes de pydantic, onde uma classe de dados recebe como input outra classe de pydantic.

In [13]:
from typing import List

class pydAsimoTeam(BaseModel):
  funcionarios: List[pydPessoa]

pydAsimoTeam(funcionarios=[
  pydPessoa(nome="Bruno", idade=32, peso=90.5),
  pydPessoa(nome="Jussara", idade=28, peso=65.0),
])

pydAsimoTeam(funcionarios=[pydPessoa(nome='Bruno', idade=32, peso=90.5), pydPessoa(nome='Jussara', idade=28, peso=65.0)])

E a validação continua funcionando mesmo com esta estrutura de nesting.

In [15]:
pydAsimoTeam(funcionarios=[Pessoa(nome="Bruno", idade=32, peso=90.5)])

ValidationError: 1 validation error for pydAsimoTeam
funcionarios.0
  Input should be a valid dictionary or instance of pydPessoa [type=model_type, input_value=<__main__.Pessoa object at 0x10f820e10>, input_type=Pessoa]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type

## Utilizando pydantic para criação de tools da OpenAI

In [16]:
import json

def obter_temperatura_atual(local, unidade="celsius"):
    if "são paulo" in local.lower():
        return json.dumps(
            {"local": "São Paulo", "temperatura": "32", "unidade": unidade}
            )
    elif "porto alegre" in local.lower():
        return json.dumps(
            {"local": "Porto Alegre", "temperatura": "25", "unidade": unidade}
            )
    else:
        return json.dumps(
            {"local": local, "temperatura": "unknown"}
            )
    
tools = [
    {
        "type": "function",
        "function": {
            "name": "obter_temperatura_atual",
            "description": "Obtém a temperatura atual em uma dada cidade",
            "parameters": {
                "type": "object",
                "properties": {
                    "local": {
                        "type": "string",
                        "description": "O nome da cidade. Ex: São Paulo",
                    },
                    "unidade": {
                        "type": "string", 
                        "enum": ["celsius", "fahrenheit"]
                    },
                },
                "required": ["local"],
            },
        },
    }
    ]


In [22]:
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

class UnidadeEnum(str, Enum):
    celsius = "celsius"
    fahrenheit = "fahrenheit"

class ObterTemperaturaAtual(BaseModel):
    """Obtém a temperatura atual de uma determinada localidade"""
    local: str = Field(description="O nome da cidade", examples=["São Paulo", "Porto Alegre"])
    unidade: Optional[UnidadeEnum]

In [24]:
from langchain_core.utils.function_calling import convert_to_openai_function

tool_temperatura = convert_to_openai_function(ObterTemperaturaAtual)
tool_temperatura

{'name': 'ObterTemperaturaAtual',
 'description': 'Obtém a temperatura atual de uma determinada localidade',
 'parameters': {'properties': {'local': {'description': 'O nome da cidade',
    'examples': ['São Paulo', 'Porto Alegre'],
    'type': 'string'},
   'unidade': {'anyOf': [{'enum': ['celsius', 'fahrenheit'],
      'title': 'UnidadeEnum',
      'type': 'string'},
     {'type': 'null'}]}},
  'required': ['local', 'unidade'],
  'type': 'object'}}

## Adicionando função externa utilizando LangChain

Agora que já sabemos criar funções que os modelos de LLM entendam, podemos passar essas funções para os modelos de linguagem através da biblioteca LangChain. Para isso temos duas formas, podemos utilizar o parâmetro functions ao chamar o método invoke dos chat_models:

In [25]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()

resposta = chat.invoke("Qual é a temperatura de Porto Alegre", functions=[tool_temperatura])
resposta

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"local":"Porto Alegre","unidade":"celsius"}', 'name': 'ObterTemperaturaAtual'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 111, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--077517c3-2182-454b-8583-32798e2b4c1a-0', usage_metadata={'input_tokens': 111, 'output_tokens': 29, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Ou podemos dar um bind e criar um novo componente de chat_model que terá acesso a função sempre que for chamado o invoke. Nestes dois casos, o modelo se comportará com o parâmetro "auto" de chamamento de função, ou seja, ele chamará a função quando necessitar, caso contratário se comportará como um modelo de linguagem normal.

In [26]:
chat = ChatOpenAI()
chat_com_func = chat.bind(functions=[tool_temperatura])
resposta = chat_com_func.invoke("Qual é a temperatura de Porto Alegre")
resposta

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"local":"Porto Alegre","unidade":"celsius"}', 'name': 'ObterTemperaturaAtual'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 111, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--fcf3e852-3386-4f04-8d06-dbcc4332a109-0', usage_metadata={'input_tokens': 111, 'output_tokens': 29, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Podemos **`obrigar`** o modelo a sempre chamar uma função da seguinte forma:

In [30]:
resposta = chat.invoke(
	"Qual é a temperatura de São Paulo", 
	functions=[tool_temperatura],
	function_call={"name": "ObterTemperaturaAtual"}
)
resposta

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"local":"São Paulo","unidade":"celsius"}', 'name': 'ObterTemperaturaAtual'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 123, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--c34a9712-8138-4ac2-88ca-e823ae720d2f-0', usage_metadata={'input_tokens': 123, 'output_tokens': 13, 'total_tokens': 136, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
resposta = chat.invoke(
	"Olá", 
	functions=[tool_temperatura],
	function_call={"name": "ObterTemperaturaAtual"}
)
resposta

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"local":"São Paulo","unidade":"celsius"}', 'name': 'ObterTemperaturaAtual'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 118, 'total_tokens': 131, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--8bf607cb-c364-4352-baf6-f6b6d0950e53-0', usage_metadata={'input_tokens': 118, 'output_tokens': 13, 'total_tokens': 131, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Adicionando a uma chain

Podemos adicionar agora este modelo com funções a um prompt e criar uma chain.

In [33]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
  ("system", "Você é um assistente amigável chamado Isaac"),
  ("user", "{input}")
])

chain = prompt | chat.bind(functions=[tool_temperatura])

In [34]:
chain.invoke({"input": "Olá"})

AIMessage(content='Olá! Como posso ajudar você hoje?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 116, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--2e8dc585-5b77-4692-8df4-d7e08b10feae-0', usage_metadata={'input_tokens': 116, 'output_tokens': 12, 'total_tokens': 128, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [35]:
chain.invoke({"input": "Qual a temperatura em Floripa?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"local":"Floripa","unidade":"celsius"}', 'name': 'ObterTemperaturaAtual'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 121, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--f02de122-aabd-4c04-959e-c75369e2b85d-0', usage_metadata={'input_tokens': 121, 'output_tokens': 27, 'total_tokens': 148, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})